<div style="padding: 20px; background: linear-gradient(90deg, #f12711 0%, #f5af19 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">📁 Module 4.2: Vector Store CRUD & Persistence</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Managing data lifecycles and metadata filtering in ChromaDB.</p>
</div>

---

## 1. Setting up Persistent Storage

In a real application, you don't want to re-embed your entire knowledge base every time the server restarts. We need **persistence**.

Chroma allows us to save the database to a local directory on our hard drive.

In [1]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
import os
import shutil

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Clean up previous runs if they exist
persist_dir = "./chroma_persistent_db"
if os.path.exists(persist_dir):
    shutil.rmtree(persist_dir)

# Initialize free local embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Initialize Chroma with a persist_directory
db = Chroma(
    collection_name="crud_demo", 
    embedding_function=embeddings,
    persist_directory=persist_dir
)

print(f"Persistent Database initialized at: {persist_dir}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\mohdf\AppData\Local\Temp\ipykernel_15156\3977379741.py:18: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(


Persistent Database initialized at: ./chroma_persistent_db


## 2. CREATE (Ingestion)
When adding documents, it is highly recommended to assign **custom IDs**. If you don't, Chroma will generate random UUIDs, making it impossible to update or delete specific documents later.

In [2]:
docs = [
    Document(page_content="Apples are red and grow on trees.", metadata={"category": "fruit", "season": "autumn"}),
    Document(page_content="Bananas are yellow and grow in tropical climates.", metadata={"category": "fruit", "season": "summer"}),
    Document(page_content="Carrots are orange root vegetables.", metadata={"category": "vegetable", "season": "all"}),
]

custom_ids = ["doc_apple", "doc_banana", "doc_carrot"]

# Add documents to DB
db.add_documents(documents=docs, ids=custom_ids)

print(f"Added {db._collection.count()} documents to the database.")

Added 3 documents to the database.


## 3. READ (Search & Metadata Filtering)
We can perform a standard similarity search, but we can also use **Metadata Filtering**. 
This tells the database: *"Only perform vector math on documents that match this exact metadata criteria."* This massively speeds up search in large databases and reduces AI hallucinations.

In [3]:
query = "Tell me about something orange."

# 1. Standard Search
standard_results = db.similarity_search_with_relevance_scores(query, k=2)
print("\n--- Standard Search ---")
for doc, score in standard_results:
    print(f"[{score:.2f}] {doc.page_content}")

# 2. Filtered Search (Only look at 'fruit')
filtered_results = db.similarity_search_with_relevance_scores(
    query, 
    k=2, 
    filter={"category": "fruit"} # Strict match requirement
)

print("\n--- Filtered Search (category=fruit) ---")
for doc, score in filtered_results:
    print(f"[{score:.2f}] {doc.page_content}")

print("Notice how the carrot was ignored in the filtered search, even though it matches the word 'orange' best!")


--- Standard Search ---
[0.18] Carrots are orange root vegetables.
[0.10] Bananas are yellow and grow in tropical climates.

--- Filtered Search (category=fruit) ---
[0.10] Bananas are yellow and grow in tropical climates.
[0.06] Apples are red and grow on trees.
Notice how the carrot was ignored in the filtered search, even though it matches the word 'orange' best!


## 4. UPDATE (Modifying existing vectors)
If a document changes in our source system, we must update its vector. Because we used custom IDs, we simply call `add_documents` again with the same ID, and it will overwrite.

In [4]:
updated_doc = Document(
    page_content="Bananas are yellow, grow in the tropics, and are rich in potassium.", 
    metadata={"category": "fruit", "season": "all"} # We updated the season too
)

# Overwrite doc_banana
db.add_documents([updated_doc], ids=["doc_banana"])

print("Successfully updated 'doc_banana'.")

# Let's verify by retrieving it directly by ID
result = db.get(ids=["doc_banana"])
print("\nVerified Content in DB:")
print(result['documents'][0])

Successfully updated 'doc_banana'.

Verified Content in DB:
Bananas are yellow, grow in the tropics, and are rich in potassium.


## 5. DELETE
Deleting vectors is crucial for compliance (e.g., GDPR data deletion requests).

In [5]:
print(f"Collection count before delete: {db._collection.count()}")

# Delete the apple document
db.delete(ids=["doc_apple"])

print(f"Collection count after delete: {db._collection.count()}")

Collection count before delete: 3
Collection count after delete: 2
